# 03 — Feature Development and Reference Contracts

This notebook records the frozen public feature boundary used by the 2015–2020 reference and TCN evaluations. Development is confined to 2015–2019; 2020 is the fixed final gate. It reports completed contract evidence rather than proposing future work.

## Leakage controls

- Targets are six demand changes at 5–30 minutes and never enter the feature tensors.
- All fitted transforms use training-fold rows only.
- AEMO histories and lags are causal; weather joins are backward-only.
- Monthly SOI changes at the preserved 19:00 Brisbane availability boundary on the first day of the following month.
- The deterministic 2021-01-01 holiday date is used only to form the 2020-12-31 day-before flag; no 2021 observation enters the RR.
- `Population_2` and `DEMANDFORECAST` are forbidden learned inputs.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import yaml
from IPython.display import display
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents) if (p / 'config' / 'public_feature_contract.yml').exists())
contract = yaml.safe_load((ROOT / 'config' / 'public_feature_contract.yml').read_text())
summary = json.loads((ROOT / 'outputs' / '03_feature_contract' / 'feature_preflight_summary.json').read_text())
display(pd.DataFrame([summary]))

## Frozen tensor routes

The exact public model uses 505 five-minute states for each temporal branch. Demand has 7 channels; system has 16; climate has 2; and each of five regions has 10. At the forecast origin, 55 encoded calendar values and one annual statewide-population scalar join at late fusion.

In [ ]:
feature_table = pd.read_csv(ROOT / 'data' / 'contracts' / 'rr_public_feature_contract_v1.csv')
display(feature_table.groupby('context', dropna=False).size().rename('expanded_features').to_frame())
display(feature_table)

## Exact reconstruction evidence

`tools/verify_rr_public_parity.py` compares the public artifact with the frozen reconstruction cache. All 96 shared frozen columns match across 629,273 public origins. The six terminal timestamps from 23:30 through 23:55 on 2020-12-31 are intentionally absent because all six future targets cannot be observed.

## Reference execution status

The established reference progression is evaluated exactly. The original five references and the three probabilistic linear references forecast the 30-minute endpoint; AEMO remains a six-horizon external benchmark. The original Ridge uses 55 base predictors, two deterministic calendar encodings, fold-fitted preprocessing, three chronological inner splits with a six-row gap, 17 candidate alpha values and the original sparse_cg implementation. Six-horizon and 30-minute endpoint comparisons are reported separately on explicit common-origin intersections.

In [ ]:
status = pd.read_csv(ROOT / 'training_output' / 'reference_models' / 'reference_execution_status.csv')
display(status)